
# 練習問題: ベクトル型で多項式を Horner 法で評価する

## 目標

ベクトル型 (`vector_size`) を使い, 多項式
`p(x) = c0 + c1*x + c2*x^2 + c3*x^3`
を配列の各要素について評価する. 計算には乗算と加算を交互に繰り返す **Horner 法** を使う:

```
acc = c3
acc = acc*x + c2
acc = acc*x + c1
acc = acc*x + c0
```

これを `nl = 8` レーンのベクトルのまま (8要素同時に) 行うのがポイントである. 各ステップ `acc = acc*x + c_k` は積和 (FMA) であり, ベクトル型で書けばコンパイラが `vfmadd...pd` のような packed double のFMA命令を生成する.

## 課題

`poly_horner.cpp` は, `x[i]` を `nl` 個ずつベクトル `xv` に読み込むところまで用意してある.
コメント `TODO` の指示に従い, **ベクトル変数 `acc` を使って Horner 法で多項式を評価**し, `acc` に `p(x)` を求めよ (係数の数だけ `acc = acc*xv + c[k]` を繰り返す).

- C++: `acc` を最高次の係数 `c[deg]` で初期化し, `k` を減らしながら `acc = acc * xv + c[k]` を繰り返す.
- Fortran: ベクトル型は無いので, 普通の `do` ループで1要素ずつ Horner 法を書く (自動ベクトル化に任せる).

## コンパイルと実行

```
# C++ (ベクトル型はプレーンなコンパイラでも使える. ここでは最適化を効かせる)
nvc++ -fast poly_horner.cpp -o poly_horner_cpp.exe
./poly_horner_cpp.exe

# Fortran
nvfortran -fast poly_horner.f90 -o poly_horner_f90.exe
./poly_horner_f90.exe
```

`n` はコマンドライン引数で指定でき (既定 64, `nl=8` の倍数), 結果はスカラ版の計算と照合される.

余裕があればアセンブリを確認せよ (`nvc++ -fast -Mkeepasm poly_horner.cpp ...` で `.s` が残る). Horner 法の各段が `vfmadd...pd` (packed double のFMA) になっていれば成功である.

## 期待される結果

- `OK: p[0]=1.000, p[63]=...` のように, ベクトル計算の結果がスカラ版と一致する.
- 実装が抜けていると `acc` が未定義のまま書き戻され, `NG` と表示される.



# 1. ツールの読み込み
- AIチュータ及びジョブ投入ツールの読み込み (カーネル起動後に一度実行すればよい)
  - `heytutor` : `%%hey` でAIチュータに質問できるようになる (使い方は末尾を参照)
  - `wisteria_submit` : `%%bash_submit` (先頭に `#PJM ...` を書く) でジョブ投入できるようになる


In [ ]:
import heytutor
import wisteria_submit

# 2. C++ ベースコード

In [ ]:
%%writefile_ poly_horner.cpp
#include <cstdio>
#include <cstdlib>

enum { nl = 8 };   /* double 8つ = 512 bit のベクトル長 */
typedef double doublev __attribute__((vector_size(sizeof(double) * nl)));

/* 多項式 p(x) = c[0] + c[1]*x + c[2]*x^2 + c[3]*x^3 */
enum { deg = 3 };
static const double c[deg + 1] = { 1.0, 2.0, 3.0, 4.0 };

/* スカラ版 (検算用) */
static double poly_ref(double x) {
  double acc = c[deg];
  for (int k = deg - 1; k >= 0; k--) acc = acc * x + c[k];
  return acc;
}

int main(int argc, char ** argv) {
  /* n は簡単のため nl の倍数とする */
  long n = (argc > 1 ? atol(argv[1]) : 64);
  double * x = (double *)malloc(sizeof(double) * n);
  double * p = (double *)malloc(sizeof(double) * n);
  for (long i = 0; i < n; i++) x[i] = 0.001 * (double)i;

  /* nl 個ずつまとめて Horner 法 acc = acc*x + c_k で多項式を評価する */
  for (long i = 0; i < n; i += nl) {
    doublev xv = *(doublev *)&x[i];
    doublev acc;
    // TODO: Horner法 acc = acc*x + c_k をベクトルのまま計算して多項式を評価せよ.
    *(doublev *)&p[i] = acc;
  }

  long err = 0;
  for (long i = 0; i < n; i++) {
    double r = poly_ref(x[i]);
    double d = p[i] - r;
    if (d < -1e-9 || d > 1e-9) err++;
  }
  if (err == 0) printf("OK: p[0]=%.3f, p[%ld]=%.3f\n", p[0], n - 1, p[n - 1]);
  else          printf("NG: %ld 要素が不正\n", err);
  free(x); free(p);
  return 0;
}

## 2-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvc++ -fast -mp=multicore poly_horner.cpp -o poly_horner_cpp.exe

## 2-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./poly_horner_cpp.exe

# 3. Fortran ベースコード

In [ ]:
%%writefile_ poly_horner.f90
! 注: ベクトル型 (vector_size) は C/C++ 独自の拡張で Fortran には無い.
! Fortran では普通のループで Horner 法を書けば, コンパイラが自動的にSIMD化する.
! 多項式 p(x) = c(1) + c(2)*x + c(3)*x^2 + c(4)*x^3
program poly_horner
  implicit none
  character(len=32) :: arg
  integer(8) :: n, i, err
  integer :: k
  integer, parameter :: deg = 3
  real(8) :: c(0:deg) = (/ 1.0d0, 2.0d0, 3.0d0, 4.0d0 /)
  real(8) :: acc, r, d
  real(8), allocatable :: x(:), p(:)
  n = 64
  if (command_argument_count() >= 1) then
     call get_command_argument(1, arg); read (arg, *) n
  end if
  allocate(x(n), p(n))
  do i = 1, n
     x(i) = 0.001d0 * real(i - 1, 8)
  end do

  ! 各要素について Horner 法 acc = acc*x + c_k で多項式を評価する
  do i = 1, n
     ! TODO: Horner法 acc = acc*x + c_k で p(i) を求めよ.
  end do

  err = 0
  do i = 1, n
     r = c(deg)
     do k = deg - 1, 0, -1
        r = r * x(i) + c(k)
     end do
     d = p(i) - r
     if (d < -1.0d-9 .or. d > 1.0d-9) err = err + 1
  end do
  if (err == 0) then
     print "(a,f0.3,a,i0,a,f0.3)", "OK: p(1)=", p(1), ", p(", n, ")=", p(n)
  else
     print "(a,i0,a)", "NG: ", err, " 要素が不正"
  end if
end program poly_horner

## 3-1. コンパイル

In [ ]:
%%bash_
PATH=/work/opt/local/x86_64/cores/nvidia/23.3/Linux_x86_64/23.3/compilers/bin:/opt/FJSVxtclanga/tcsds-1.2.41/bin:$PATH
nvfortran -fast -mp=multicore poly_horner.f90 -o poly_horner_f90.exe

## 3-2. 実行
- 計算ノードにジョブとして投入して実行する。スレッド数・キュー・制限時間は `#PJM` 行で調整する。
- すぐにログインノードで試したいときは, 先頭の `%%bash_submit` を `%%bash_` に書き換えて (`#PJM` 行はコメントなので無視される) 実行すればよい。ただし数秒で終わる軽いジョブに限る。

In [ ]:
%%bash_submit
#PJM -L rscgrp=lecture-a
#PJM -L elapse=0:01:00
#PJM -L gpu=1
#PJM --omp thread=9
#PJM -g gt69
#PJM -j
#PJM -o 0output.txt

./poly_horner_f90.exe


# 4. AIチュータへの質問の仕方 (参考)
- 先頭で `import heytutor` 済みなら, セルに `%%hey` と書いて質問できる。
- ChatGPTなどと同様に自由に質問してよい。ただし「答えをそのまま教えて」などは自制すること。
- セル内で使える変数 (自動で展開される):
  - `{file:FILENAME}` : _FILENAME_ の中身 (例: `{file:problem.md}`, `{file:poly_horner.cpp}`)
  - `{bash[-1]}` : 最後に実行した `%%bash_` セルの入力・出力, `{bash[-2]}` = その前, ...
- 以下は質問例 (必要に応じてコピーして使う。Fortranなら `.cpp` を `.f90` に書き換える)。

## 4-1. 一般的な質問

In [ ]:
%%hey

C++の関数定義の文法どんなだっけ?

## 4-2. この問題に関するヒント

In [ ]:
%%hey

この問題に関するヒントを教えて

問題:
{file:problem.md}

## 4-3. 困ったときのヘルプ
- コンパイル時や実行時のエラー直後に実行するとエラーに関するヘルプが得られる。

In [ ]:
%%hey

以下のエラーが出た。何が間違い?

プログラム:
{file:poly_horner.cpp}

コマンドと実行結果:
{bash[-1]}

## 4-4. フィードバック
- 答えが出た後も, 無駄なところやより良いやり方がないかを聞くことを推奨。

In [ ]:
%%hey

私の答に対するフィードバックをください。

問題:
{file:problem.md}

私の答:
{file:poly_horner.cpp}